# KAE Multi-Model Agent (Colab)

Один Colab-GPU агент обслуживает несколько задач **одной моделью Qwen2.5-VL-7B**
(мультимодальная: таблицы, формулы, схемы) — задача выбирается по `task`, меняется
только промпт. Единое окружение transformers, без конфликта версий.

| task | что делает | роль в KAE |
|---|---|---|
| `table`   | таблица → LaTeX tabular   | table   |
| `formula` | формулы → LaTeX           | formula |
| `vision`  | схема → TikZ              | vision  |

HTTP: `GET /health` + `POST /infer {image_b64, task, prompt?}`.
GOT-OCR2.0 (топ-качество таблиц, но transformers 4.37) — отдельным агентом при желании.

**Перед запуском:** Runtime → Change runtime type → **T4 GPU** (не TPU).

In [ ]:
# 1. GPU + зависимости (torch НЕ трогаем — оставляем CUDA-torch из Kaggle/Colab).
# bitsandbytes не нужен: на Kaggle 2×T4 гоним fp16 с шардингом (device_map='auto').
!nvidia-smi -L || print('Поставь GPU: Kaggle Settings -> Accelerator -> GPU T4 x2')
!pip -q install transformers==4.49.0 qwen-vl-utils accelerate flask
import torch
assert torch.cuda.is_available(), 'CUDA недоступна! Restart session, затем Run all'
print(f'CUDA OK: {torch.cuda.device_count()} × {torch.cuda.get_device_name(0)}')

In [ ]:
# 2. Qwen2.5-VL-7B — fp16, шардинг по всем видимым GPU (используем оба T4 на Kaggle).
# max_pixels ограничивает разрешение visual-энкодера; на 2×T4 можем позволить больше.
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
n_gpu = torch.cuda.device_count()
print(f'GPU: {n_gpu} × {torch.cuda.get_device_name(0)}')
MODEL = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    'Qwen/Qwen2.5-VL-7B-Instruct',
    torch_dtype=torch.float16,
    device_map='auto',
)
PROC = AutoProcessor.from_pretrained(
    'Qwen/Qwen2.5-VL-7B-Instruct',
    min_pixels=256 * 28 * 28,           # ~200k
    max_pixels=(1280 if n_gpu >= 2 else 768) * 28 * 28,
)
print('Qwen2.5-VL готов, max_pixels =', PROC.image_processor.max_pixels)

In [ ]:
# 3. Инференс по задаче (меняется только промпт)
from qwen_vl_utils import process_vision_info
PROMPTS = {
    'table':   'Convert the table in this image to a LaTeX tabular environment, '
               'preserving merged cells and all values. Output ONLY the LaTeX.',
    'formula': 'Extract every mathematical formula from this image as LaTeX. '
               'Output ONLY the LaTeX.',
    'vision':  'Reconstruct this block diagram as a LaTeX tikzpicture: boxes with '
               'their text, titles above boxes, and connecting arrows. '
               'Output ONLY the tikzpicture environment.',
}
def infer(image_path, task='table', prompt=None):
    prompt = prompt or PROMPTS.get(task, PROMPTS['table'])
    msgs = [{'role': 'user', 'content': [
        {'type': 'image', 'image': image_path}, {'type': 'text', 'text': prompt}]}]
    text = PROC.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    imgs, vids = process_vision_info(msgs)
    inputs = PROC(text=[text], images=imgs, videos=vids, padding=True, return_tensors='pt').to('cuda')
    out = MODEL.generate(**inputs, max_new_tokens=2048)
    trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, out)]
    return PROC.batch_decode(trimmed, skip_special_tokens=True)[0]
print('infer() готов')

In [ ]:
# 4. HTTP-сервис: /health + /infer (+ /ocr для обратной совместимости)
from flask import Flask, request, jsonify
import base64, tempfile, threading, os
app = Flask(__name__)

@app.get('/health')
def health():
    return jsonify({'status': 'ok', 'kind': 'multimodel', 'model': 'Qwen2.5-VL-7B',
                    'tasks': ['table', 'formula', 'vision']})

def _run(task):
    d = request.get_json(force=True)
    raw = base64.b64decode(d['image_b64'])
    fd, p = tempfile.mkstemp(suffix='.png'); os.write(fd, raw); os.close(fd)
    try:
        return jsonify({'text': infer(p, task=d.get('task', task), prompt=d.get('prompt'))})
    finally:
        os.remove(p)

@app.post('/infer')
def _infer(): return _run('table')

@app.post('/ocr')
def _ocr(): return _run('table')

PORT = 5005
threading.Thread(target=lambda: app.run(host='0.0.0.0', port=PORT), daemon=True).start()
print(f'Сервис на :{PORT} (/health, /infer, /ocr)')

In [ ]:
# 5. Туннель наружу — публичный URL для менеджера агентов KAE
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
import subprocess, re, itertools
proc = subprocess.Popen(['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{PORT}'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in itertools.islice(proc.stdout, 200):
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0); break
print('\n' + '=' * 60)
print('AGENT URL:', url)
print('kind=multimodel, роли: table, formula, vision')
print('=' * 60)
import time
while True: time.sleep(300)